In [1]:
#i/o

from pathlib import Path

import dask.dataframe as dd


class XeniumBundle:
    def __init__(self, path: Path):
        self.path = Path(path)

    def validate(self) -> None:
        if not self.path.exists():
            raise FileNotFoundError(f"Xenium output bundle not found: {self.path}")
        if not self.path.is_dir():
            raise ValueError(f"Expected a Xenium output bundle directory: {self.path}")
        transcript_file = self.path / "transcripts.parquet"
        if not transcript_file.exists():
            raise FileNotFoundError(f"transcripts.parquet not found in Xenium bundle: {self.path}")

    @property
    def transcript_file(self) -> Path:
        return self.path / "transcripts.parquet"

    def load_dataframe(self) -> dd.DataFrame:
        self.validate()
        try:
            return dd.read_parquet(self.transcript_file)
        except Exception as e:
            raise ValueError(f"Unable to read {self.transcript_file} as a parquet file.") from e

In [2]:
#subsample

import dask.dataframe as dd
import numpy as np
import pandas as pd

from subqcat.io import XeniumBundle


class SampleXenium:
    def __init__(self, xenium: XeniumBundle):
        self.xenium = xenium

    def clean_data(self) -> dd.DataFrame:
        df = self.xenium.load_dataframe()
        required_cols = ['qv', 'is_gene', 'cell_id', 'codeword_index', 'codeword_category']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"The dataset is missing required columns for cleaning: {missing_cols}")
        df_quality = df[df['qv']>= 20.0]
        df_gene = df_quality[df_quality['is_gene']==True]
        df_clean = df_gene[df_gene['cell_id'] != "UNASSIGNED"]
        df_clean = df_clean[df_clean['cell_id'] != "-1"]
        df = df_clean.drop(columns=["codeword_index", "codeword_category", "is_gene"])
        return df

    def subsample(self) -> pd.DataFrame:
        clean_df = self.clean_data()
        unique_cells = clean_df['cell_id'].unique().compute()
        if len(unique_cells) == 0:
            raise ValueError("No cells remain after cleaning the data! Cannot subsample.")
        np.random.seed(42)
        sample_size = min(5000, len(unique_cells))
        sampled_cell_ids = np.random.choice(unique_cells, size=sample_size, replace=False)
        df_subsampled = clean_df[clean_df['cell_id'].isin(sampled_cell_ids)]
        df_final = df_subsampled.compute()
        return df_final


In [3]:
bundle = XeniumBundle("/Users/priyaltripathi/SubQCAT/data/Xenium_Prime_Mouse_Brain_Coronal_FF_outs")
df = SampleXenium(bundle)
subsampled_df = df.subsample()

In [4]:
subsampled_df.head()


,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance
3965,281758444559544,kiiegcla-1,0,A1cf,130.171875,3876.437500,27.937500,40.0,F6,3.796875
3969,281758444569922,kiiegcla-1,1,Aatf,125.250000,3882.453125,28.421875,40.0,F6,0.000000
3971,281758444583380,kiiegcla-1,0,Abca3,127.078125,3879.109375,28.093750,40.0,F6,1.859375
3973,281758444594794,kiiegcla-1,0,Abca3,126.265625,3887.453125,22.656250,40.0,F6,1.562500
3974,281758444615853,kiiegcla-1,0,Abca7,122.234375,3885.203125,25.000000,26.5,F6,1.359375


In [14]:
df_grouped_by_cell = subsampled_df.groupby('cell_id')
centroids = df_grouped_by_cell[['x_location', 'y_location', 'z_location']].transform('mean')
subsampled_df['centroid_x'] = centroids['x_location']
subsampled_df['centroid_y'] = centroids['y_location']
subsampled_df['centroid_z'] = centroids['z_location']

subsampled_df.head()

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance,centroid_x,centroid_y,centroid_z
3965,281758444559544,kiiegcla-1,0,A1cf,130.171875,3876.437500,27.937500,40.0,F6,3.796875,124.594193,3879.578857,25.452757
3969,281758444569922,kiiegcla-1,1,Aatf,125.250000,3882.453125,28.421875,40.0,F6,0.000000,124.594193,3879.578857,25.452757
3971,281758444583380,kiiegcla-1,0,Abca3,127.078125,3879.109375,28.093750,40.0,F6,1.859375,124.594193,3879.578857,25.452757
3973,281758444594794,kiiegcla-1,0,Abca3,126.265625,3887.453125,22.656250,40.0,F6,1.562500,124.594193,3879.578857,25.452757
3974,281758444615853,kiiegcla-1,0,Abca7,122.234375,3885.203125,25.000000,26.5,F6,1.359375,124.594193,3879.578857,25.452757


In [16]:
euclidean_formula = np.sqrt((subsampled_df['x_location'] - subsampled_df['centroid_x'])**2 + ((subsampled_df['y_location'] - subsampled_df['centroid_y'])**2) + (subsampled_df['z_location'] - subsampled_df['centroid_z'])**2)
subsampled_df["distance_from_center"] = euclidean_formula
subsampled_df.head()

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance,centroid_x,centroid_y,centroid_z,distance_from_center
3965,281758444559544,kiiegcla-1,0,A1cf,130.171875,3876.437500,27.937500,40.0,F6,3.796875,124.594193,3879.578857,25.452757,6.866776
3969,281758444569922,kiiegcla-1,1,Aatf,125.250000,3882.453125,28.421875,40.0,F6,0.000000,124.594193,3879.578857,25.452757,4.184156
3971,281758444583380,kiiegcla-1,0,Abca3,127.078125,3879.109375,28.093750,40.0,F6,1.859375,124.594193,3879.578857,25.452757,3.655842
3973,281758444594794,kiiegcla-1,0,Abca3,126.265625,3887.453125,22.656250,40.0,F6,1.562500,124.594193,3879.578857,25.452757,8.521633
3974,281758444615853,kiiegcla-1,0,Abca7,122.234375,3885.203125,25.000000,26.5,F6,1.359375,124.594193,3879.578857,25.452757,6.116054


In [19]:
df_grouped_by_cell = subsampled_df.groupby('cell_id')
mean_distance = df_grouped_by_cell[['distance_from_center']].mean()
mean_distance


,distance_from_center
cell_id,
aabagmcb-1,3.254280
aabbijjj-1,3.320385
aabhojob-1,2.788721
aabiojde-1,2.434427
aabkfjcm-1,3.072032
...,...
oiidghgi-1,4.372100
oijgkpjh-1,5.183146
oikenajf-1,2.863241


In [ ]:
# from sklearn.cluster import KMeans
# from scipy.spatial.distance import cdist
# import matplotlib.pyplot as plt
# from sklearn import metrics

# distortions = []
# inertias = []
# mapping1 = {}
# mapping2 = {}
# K = range(1, 10)

# for k in K:
#     kmeanModel = KMeans(n_clusters=k, random_state=42).fit(distance_array)
    
#     distortions.append(sum(np.min(cdist(distance_array, kmeanModel.cluster_centers_, 'euclidean'), axis=1)**2) / distance_array.shape[0])
    
#     inertias.append(kmeanModel.inertia_)
    
#     mapping1[k] = distortions[-1]
#     mapping2[k] = inertias[-1]

# print("Distortion values:")
# for key, val in mapping1.items():
#     print(f'{key} : {val}')

# plt.plot(K, distortions, 'bx-')
# plt.xlabel('Number of Clusters (k)')
# plt.ylabel('Distortion')
# plt.title('The Elbow Method using Distortion')
# plt.show()

# print("Inertia values:")
# for key, val in mapping2.items():
#     print(f'{key} : {val}')

# plt.plot(K, inertias, 'bx-')
# plt.xlabel('Number of Clusters (k)')
# plt.ylabel('Inertia')
# plt.title('The Elbow Method using Inertia')
# plt.show()


In [ ]:
from sklearn.cluster import KMeans
distance_array = np.array(mean_distance['distance_from_center']).reshape(-1, 1)
kmeanModel = KMeans(n_clusters=2, random_state=42).fit(distance_array)
clusters = kmeanModel.labels_
mean_distance['spatial_cluster'] = clusters

In [32]:
mean_distance.reset_index()

,cell_id,distance_from_center,spatial_cluster
0,aabagmcb-1,3.254280,0
1,aabbijjj-1,3.320385,0
2,aabhojob-1,2.788721,0
3,aabiojde-1,2.434427,0
4,aabkfjcm-1,3.072032,0
...,...,...,...
4995,oiidghgi-1,4.372100,1
4996,oijgkpjh-1,5.183146,1
4997,oikenajf-1,2.863241,0
4998,oilcalkg-1,2.653394,0


In [29]:
cells_df = dd.read_parquet("/Users/priyaltripathi/SubQCAT/data/Xenium_Prime_Mouse_Brain_Coronal_FF_outs/cells.parquet")
cells_df.head()

,cell_id,x_centroid,y_centroid,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,nucleus_count,segmentation_method
0,aaaalabd-1,610.889465,1741.865845,2193,1,0,0,3,2,2199,49.626721,33.099532,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
1,aaabcnhh-1,622.179077,1794.472900,1873,0,0,0,0,0,1873,55.226096,38.427970,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
2,aaacnmip-1,615.411621,1798.880859,2163,0,0,1,1,0,2165,85.796878,75.456096,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
3,aaadikih-1,613.914185,1845.295044,2441,0,0,0,4,1,2446,94.060472,52.290939,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
4,aaaeinek-1,739.221436,3268.997070,1407,0,0,0,0,0,1407,112.258442,NaN,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...


In [35]:
mean_distance = mean_distance.reset_index()

In [37]:
chosen_cells = mean_distance['cell_id'].unique()
filtered_dask_cells = cells_df[cells_df['cell_id'].isin(chosen_cells)].compute()


In [39]:
print(type(mean_distance))

<class 'pandas.DataFrame'>


In [40]:
final_master_df = pd.merge(mean_distance, filtered_dask_cells, on='cell_id', how='inner')


In [43]:
print(final_master_df.groupby('spatial_cluster')[['distance_from_center', 'transcript_counts', 'cell_area']].mean())


                 distance_from_center  transcript_counts   cell_area
spatial_cluster                                                     
0                            2.977651         750.635306   53.479909
1                            4.781469        2305.845884  145.355752
